01_data_preparation

02_feature_extraction_and_ml_traning

03_acoustic_DL

04_linguistic_DL

05_image_DL

# Check CPU/GPU/Memory

In [18]:
# In order to use a GPU with this notebook, select the Runtime > Change runtime type menu and then set the hardware accelerator
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Fri Jun 12 09:11:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   47C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [19]:
# If the execution result of running the code cell below is 'Not using a high-RAM runtime',
# then we can enable a high-RAM runtime via Runtime > Change runtime type in the menu.
# Then select High-RAM in the Runtime shape toggle button

import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 56.9 gigabytes of available RAM

You are using a high-RAM runtime!


# Imports

In [ ]:
!pip install google-cloud-storage librosa scikit-learn 

In [21]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split



In [22]:
# ML Model imports
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Configurations

In [ ]:
from google.colab import auth
auth.authenticate_user()

# To find the credentials automatically
from google.cloud import storage
client = storage.Client(project=GCP_PROJECT_ID) # GCP Project ID

In [ ]:
# Initialize GCP Storage
client = storage.Client()
bucket_name = VOICE_DATA_BUCKET_NAME # GCP Storage bucket name
bucket = client.get_bucket(bucket_name)
print(f"Bucket {bucket.name} successfully loaded.")

Bucket voicedata-bucket successfully loaded.


In [25]:

print(os.getcwd())

/content


In [26]:
# Configuration

# GCS Paths
ADDRESSO_PREFIX_AUDIO = 'ADDReSS2021/processed/audio/'
ADDRESSO_CSV_PATH = 'ADDReSS2021/processed/final_transcripts.csv'

WLS_PREFIX_AUDIO = 'WLS/processed/audio/'
WLS_CSV_PATH = 'WLS/processed/final_transcripts.csv'


CATEGORIES = ['CN', 'AD']

# Ensure local directories exist for processing
os.makedirs("local_data/ADDReSS2021/processed_audio", exist_ok=True) # ADDReSSo audio
os.makedirs("local_data/WLS/processed_audio", exist_ok=True)       # WLS audio

# Download ADDReSSo dataset from GCS

In [27]:
# Download the final transcripts CSV
print("Downloading transcripts...")
csv_blob = bucket.blob(ADDRESSO_CSV_PATH)
csv_blob.download_to_filename("local_data/ADDReSS2021/final_transcripts.csv")

# Load into a pandas DataFrame
df = pd.read_csv("local_data/ADDReSS2021/final_transcripts.csv")
print(f"\nSuccessfully Loaded {len(df)} records.")
display(df.head())


Successfully Loaded 161 records.


,participant_id,audio_path,transcript,label,label_numeric
0,adrso054,local_data/processed_audio/adrso054_par.wav,I'm gonna keep burying the meal. I'm gonna hav...,AD,1
1,adrso007,local_data/processed_audio/adrso007_par.wav,Our mother standing by the sink kind of looks ...,CN,0
2,adrso128,local_data/processed_audio/adrso128_par.wav,"Um, Cookie John, and he's stepping on us, well...",AD,1
3,adrso309,local_data/processed_audio/adrso309_par.wav,"Mother is, um, drying the dishes, looking out ...",CN,0
4,adrso274,local_data/processed_audio/adrso274_par.wav,"Alright, the boy is taking a cookie out of the...",CN,0


In [28]:
# Download the processed participant audio files
print("Downloading ADDReSSo audio files from GCS...")

addresso_blobs = bucket.list_blobs(prefix=ADDRESSO_PREFIX_AUDIO)

for blob in addresso_blobs:
    if blob.name.endswith(".wav"):
        file_name = os.path.basename(blob.name)
        local_path = f"local_data/ADDReSS2021/processed_audio/{file_name}"

        # Only download if it doesn't already exist locally
        if not os.path.exists(local_path):
            blob.download_to_filename(local_path)


# Download WLS Dataset from GCS

In [29]:
# Download the final transcripts CSV
print("Downloading transcripts...")
csv_blob_wls = bucket.blob(WLS_CSV_PATH)
csv_blob_wls.download_to_filename("local_data/WLS/final_transcripts.csv")

# Load into a pandas DataFrame
df_wls = pd.read_csv("local_data/WLS/final_transcripts.csv")
print(f"Success! Loaded {len(df_wls)} records.")
display(df_wls.head())

Success! Loaded 40 records.


,participant_id,audio_path,transcript,label,label_numeric
0,137,local_data/processed_audio/00137_par.wav,"xxx boom . a lady doing dishes , the water run...",CN,0
1,399,local_data/processed_audio/00399_par.wav,"girl , boy . mother . water running out_of the...",CN,0
2,1362,local_data/processed_audio/01362_par.wav,ah a kid stealing cookies . and the stool is g...,CN,0
3,1379,local_data/processed_audio/01379_par.wav,everything I see . you mean from a safety aspe...,CN,0
4,1748,local_data/processed_audio/01748_par.wav,well ‡ the kids are trying to get into the coo...,CN,0


In [30]:
# Download the processed participant audio files
print("Downloading WLS audio files from GCS...")

wls_blobs = bucket.list_blobs(prefix=WLS_PREFIX_AUDIO)

for blob in wls_blobs:
    if blob.name.endswith(".wav"):
        file_name = os.path.basename(blob.name)
        local_path = f"local_data/WLS/processed_audio/{file_name}"

        # Only download if it doesn't already exist locally
        if not os.path.exists(local_path):
            blob.download_to_filename(local_path)

In [31]:
print("--- Fixing Audio Paths in DataFrames ---")

# 1. Update the ADDReSSo paths to include the ADDReSS2021 folder
df['audio_path'] = df['audio_path'].str.replace(
    'local_data/processed_audio/',
    'local_data/ADDReSS2021/processed_audio/'
)

# 2. Update the WLS paths to include the WLS folder
df_wls['audio_path'] = df_wls['audio_path'].str.replace(
    'local_data/processed_audio/',
    'local_data/WLS/processed_audio/'
)

print("Success! Paths updated. Here is a sample:")
display(df[['participant_id', 'audio_path']].head(2))

--- Fixing Audio Paths in DataFrames ---
Success! Paths updated. Here is a sample:


,participant_id,audio_path
0,adrso054,local_data/ADDReSS2021/processed_audio/adrso05...
1,adrso007,local_data/ADDReSS2021/processed_audio/adrso00...


# ADDReSSo dataset - Train/Validation/Test Split

In [32]:
print("--- Shuffling Datasets ---")

# Explicitly shuffle the entire DataFrames and reset their indices
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df_wls = df_wls.sample(frac=1, random_state=42).reset_index(drop=True)

--- Shuffling Datasets ---


In [33]:

print("--- Data Splitting ---")
# 1. ADDReSSo Split
X_addresso = df[['participant_id', 'audio_path', 'transcript']]
y_addresso = df['label']

# First Split: 70% Train, 30% Temp (which will become Val/Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_addresso, y_addresso, test_size=0.30, stratify=y_addresso, random_state=42
)

# Second Split: Divide the 30% Temp into 15% Validation and 15% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

# 2. WLS Setup (No split, used entirely for generalization testing)
X_wls = df_wls[['participant_id', 'audio_path', 'transcript']]
y_wls = df_wls['label']

print(f"ADDReSSo Training Set (70%):   {X_train.shape[0]} samples")
print(f"ADDReSSo Validation Set (15%): {X_val.shape[0]} samples")
print(f"ADDReSSo Testing Set (15%):    {X_test.shape[0]} samples")
print(f"WLS External Dataset:          {X_wls.shape[0]} samples")

--- Data Splitting ---
ADDReSSo Training Set (70%):   112 samples
ADDReSSo Validation Set (15%): 24 samples
ADDReSSo Testing Set (15%):    25 samples
WLS External Dataset:          40 samples


#Feature Extraction

- extract the numerical representations required for traditional ML models training.

## 1. Audio (Acoustic Features) - Librosa

In [34]:
import gc
import librosa
import warnings

# Suppress librosa warnings about empty/short audio files to keep logs clean
warnings.filterwarnings('ignore', module='librosa')

print("--- Extracting Acoustic Features (Librosa) ---")
os.makedirs("local_data/features", exist_ok=True)

def extract_librosa_features(file_path):
    """Loads a single audio file and extracts aggregated librosa features."""
    # Load audio (standardizing sample rate to 16kHz is common practice)
    y, sr = librosa.load(file_path, sr=16000)

    # If the file is completely empty, raise an error to trigger the except block
    if len(y) == 0:
        raise ValueError("Audio file is empty.")

    # 1. Extract 2D Time-Series Features
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    delta_mfcc = librosa.feature.delta(mfcc)
    delta2_mfcc = librosa.feature.delta(mfcc, order=2)
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)

    features = {}

    # 2. Helper function to calculate Mean and Std Dev across the time axis (axis=1)
    def add_stats(feature_name, feature_array):
        means = np.mean(feature_array, axis=1)
        stds = np.std(feature_array, axis=1)
        for i, (m, s) in enumerate(zip(means, stds)):
            features[f"{feature_name}_{i}_mean"] = m
            features[f"{feature_name}_{i}_std"] = s

    # 3. Aggregate and store all features
    add_stats("mfcc", mfcc)
    add_stats("delta_mfcc", delta_mfcc)
    add_stats("delta2_mfcc", delta2_mfcc)
    add_stats("centroid", centroid)
    add_stats("contrast", contrast)
    add_stats("chroma", chroma)

    # Attach the file path so we can set it as the index later
    features['file'] = file_path
    return features


def extract_features_in_batches(file_paths, output_csv, batch_size=20):
    """Processes audio files in chunks using librosa and saves to disk."""
    total_files = len(file_paths)

    if os.path.exists(output_csv):
        os.remove(output_csv)

    print(f"Starting extraction for {total_files} files (Batch size: {batch_size})...")

    for chunk_start in range(0, total_files, batch_size):
        chunk_end = min(chunk_start + batch_size, total_files)
        batch_files = file_paths[chunk_start:chunk_end]

        temp_features = []
        for file_path in batch_files:
            try:
                features_dict = extract_librosa_features(file_path)
                temp_features.append(features_dict)
            except Exception as e:
                print(f"  [Skipped] {os.path.basename(file_path)}: {e}")

        if temp_features:
            # Convert the list of dictionaries to a DataFrame
            batch_df = pd.DataFrame(temp_features)

            # Set the 'file' column as the index to mimic OpenSMILE's structure
            batch_df.set_index('file', inplace=True)

            write_header = not os.path.exists(output_csv)
            batch_df.to_csv(output_csv, mode='a', header=write_header)

            del temp_features
            del batch_df
            gc.collect()

        print(f"  [{chunk_end}/{total_files}] files processed and saved.")

    print("Loading compiled dataset back into memory...")
    # Load back with 'file' as the index column
    return pd.read_csv(output_csv, index_col='file')

--- Extracting Acoustic Features (Librosa) ---


In [35]:
# --- Run the Batch Processing ---

print("\n--- Processing ADDReSSo Training Data ---")
X_train_audio = extract_features_in_batches(
    X_train['audio_path'].tolist(),
    "local_data/features/X_train_audio.csv",
    batch_size=20
)

print("\n--- Processing ADDReSSo Validation Data ---")
X_val_audio = extract_features_in_batches(
    X_val['audio_path'].tolist(),
    "local_data/features/X_val_audio.csv",
    batch_size=20
)

print("\n--- Processing ADDReSSo Testing Data ---")
X_test_audio = extract_features_in_batches(
    X_test['audio_path'].tolist(),
    "local_data/features/X_test_audio.csv",
    batch_size=20
)

print("\n--- Processing WLS External Testing Data ---")
X_wls_audio = extract_features_in_batches(
    X_wls['audio_path'].tolist(),
    "local_data/features/X_wls_audio.csv",
    batch_size=20
)


--- Processing ADDReSSo Training Data ---
Starting extraction for 112 files (Batch size: 20)...
  [20/112] files processed and saved.
  [40/112] files processed and saved.
  [60/112] files processed and saved.
  [80/112] files processed and saved.
  [100/112] files processed and saved.
  [112/112] files processed and saved.
Loading compiled dataset back into memory...

--- Processing ADDReSSo Validation Data ---
Starting extraction for 24 files (Batch size: 20)...
  [20/24] files processed and saved.
  [24/24] files processed and saved.
Loading compiled dataset back into memory...

--- Processing ADDReSSo Testing Data ---
Starting extraction for 25 files (Batch size: 20)...
  [20/25] files processed and saved.
  [25/25] files processed and saved.
Loading compiled dataset back into memory...

--- Processing WLS External Testing Data ---
Starting extraction for 40 files (Batch size: 20)...
  [20/40] files processed and saved.
  [40/40] files processed and saved.
Loading compiled dataset

In [36]:
print(f"\nSuccess! Final Training Feature set Shape: {X_train_audio.shape}")
print(f"Success! Final Validation Feature set Shape:  {X_val_audio.shape}")
print(f"Success! Final Testing Feature set Shape:  {X_test_audio.shape}")

print(f"Success! Final External Testing Feature set Shape:  {X_wls_audio.shape}")



Success! Final Training Feature set Shape: (112, 160)
Success! Final Validation Feature set Shape:  (24, 160)
Success! Final Testing Feature set Shape:  (25, 160)
Success! Final External Testing Feature set Shape:  (40, 160)


### Verification

In [37]:
# 1. Print the total count and a sample of the column names
print(f"Total engineered features per participant: {len(X_train_audio.columns)}")
print("\nSample feature names (first 10):")
print(X_train_audio.columns[:10].tolist())

# 2. Display a snapshot of the actual data
# (Showing the first 5 participants and the first 8 features)
print("\nSample of the extracted data:")
display(X_train_audio.iloc[:5, :8])

Total engineered features per participant: 160

Sample feature names (first 10):
['mfcc_0_mean', 'mfcc_0_std', 'mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std']

Sample of the extracted data:


,mfcc_0_mean,mfcc_0_std,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,mfcc_3_mean,mfcc_3_std
file,,,,,,,,
local_data/ADDReSS2021/processed_audio/adrso261_par.wav,-296.93405,26.665127,73.17506,16.379112,15.614894,9.864286,14.388232,11.077040
local_data/ADDReSS2021/processed_audio/adrso056_par.wav,-429.48270,49.644634,83.95873,38.614822,18.448248,22.050217,12.002615,15.297336
local_data/ADDReSS2021/processed_audio/adrso059_par.wav,-324.29773,141.540280,89.43388,67.905710,13.944099,28.261120,13.396301,26.328665
local_data/ADDReSS2021/processed_audio/adrso138_par.wav,-359.15436,114.234825,80.11313,64.409610,5.009028,31.390390,4.932878,22.765059
local_data/ADDReSS2021/processed_audio/adrso302_par.wav,-306.22552,112.973495,93.88418,64.830300,8.223101,31.356165,8.987194,26.287040


### Align Datasets, if any corrupted files were removed during feature extraction

In [38]:
print("--- Aligning Datasets with Successful Audio Features ---")

def align_data(X_orig, y_orig, X_audio, dataset_name):
    orig_len = len(X_orig)
    successful_files = X_audio.index

    # Filter X and y based on successful audio extraction
    X_clean = X_orig[X_orig['audio_path'].isin(successful_files)]
    y_clean = y_orig.loc[X_clean.index]

    # Print the verification for this specific split
    print(f"Original {dataset_name:<11} Patients: {orig_len:3d} -> Cleaned: {len(X_clean)}")
    return X_clean, y_clean

# Process all 4 datasets through the function
X_train, y_train = align_data(X_train, y_train, X_train_audio, "Training")
X_val, y_val = align_data(X_val, y_val, X_val_audio, "Validation")
X_test, y_test = align_data(X_test, y_test, X_test_audio, "Testing")
X_wls, y_wls = align_data(X_wls, y_wls, X_wls_audio, "WLS")

# Print the final aligned shapes
print("-" * 45)
print(f"Aligned X_train_audio Shape: {X_train_audio.shape} | y_train Shape: {y_train.shape}")
print(f"Aligned X_val_audio Shape:   {X_val_audio.shape} | y_val Shape:   {y_val.shape}")
print(f"Aligned X_test_audio Shape:  {X_test_audio.shape} | y_test Shape:  {y_test.shape}")
print(f"Aligned X_wls_audio Shape:   {X_wls_audio.shape} | y_wls Shape:   {y_wls.shape}")

--- Aligning Datasets with Successful Audio Features ---
Original Training    Patients: 112 -> Cleaned: 112
Original Validation  Patients:  24 -> Cleaned: 24
Original Testing     Patients:  25 -> Cleaned: 25
Original WLS         Patients:  40 -> Cleaned: 40
---------------------------------------------
Aligned X_train_audio Shape: (112, 160) | y_train Shape: (112,)
Aligned X_val_audio Shape:   (24, 160) | y_val Shape:   (24,)
Aligned X_test_audio Shape:  (25, 160) | y_test Shape:  (25,)
Aligned X_wls_audio Shape:   (40, 160) | y_wls Shape:   (40,)


### Saving the Extracted Acoustic Features in GCS

In [39]:
# Define the upload function (used by both acoustic and linguistic feature sets)
def upload_feature_to_gcs(local_file, gcs_blob_name):
    """Uploads a file to the GCS bucket."""
    blob = bucket.blob(gcs_blob_name)
    blob.upload_from_filename(local_file)
    print(f"  Uploaded -> {gcs_blob_name}")

In [40]:
# Helper for saving and tracking AUDIO uploads
uploads_audio = []
os.makedirs("local_data/features/librosa", exist_ok=True)

def save_and_queue_audio(name, audio_df, labels_df, dataset_prefix):
    # Save Audio CSV (index=True to keep the file paths)
    audio_path = f"local_data/features/librosa/{name}_audio.csv"
    audio_df.to_csv(audio_path, index=True)
    uploads_audio.append((audio_path, f"{dataset_prefix}/features/librosa/{name}_audio.csv"))

    # Save Labels (index=True to keep the alignment keys)
    label_path = f"local_data/features/librosa/{name}_labels.csv"
    labels_df.to_csv(label_path, index=True)
    uploads_audio.append((label_path, f"{dataset_prefix}/features/librosa/{name}_labels.csv"))


# Queue ADDReSSo (Audio + Labels)
save_and_queue_audio("X_train", X_train_audio, y_train, "ADDReSS2021")
save_and_queue_audio("X_val", X_val_audio, y_val, "ADDReSS2021")
save_and_queue_audio("X_test", X_test_audio, y_test, "ADDReSS2021")

# Queue WLS (Audio + Labels)
save_and_queue_audio("X_wls", X_wls_audio, y_wls, "WLS")

# Upload all audio and label files
for local_file, gcs_path in uploads_audio:
    upload_feature_to_gcs(local_file, gcs_path)

print("\nSuccess! Acoustic features and aligned labels for ADDReSSo and WLS have been safely backed up to GCS.")

  Uploaded -> ADDReSS2021/features/librosa/X_train_audio.csv
  Uploaded -> ADDReSS2021/features/librosa/X_train_labels.csv
  Uploaded -> ADDReSS2021/features/librosa/X_val_audio.csv
  Uploaded -> ADDReSS2021/features/librosa/X_val_labels.csv
  Uploaded -> ADDReSS2021/features/librosa/X_test_audio.csv
  Uploaded -> ADDReSS2021/features/librosa/X_test_labels.csv
  Uploaded -> WLS/features/librosa/X_wls_audio.csv
  Uploaded -> WLS/features/librosa/X_wls_labels.csv

Success! Acoustic features and aligned labels for ADDReSSo and WLS have been safely backed up to GCS.


##2. Text (Linguistic Features)

Research shows that Mean Length of Utterance (MLU), Type-Token Ratio (TTR), and Long Pause Ratio (LPR) as sensitive markers for cognitive decline.

Lexical: TTR, MATTR, Brunet, Honore

Syntactic: Mean sentence length, Dependency depth, POS ratios

Semantic: Word frequency, Embedding coherence

In [41]:
print(f"Aligned X_train Shape: {X_train.shape}")
print(f"Aligned X_val Shape:   {X_val.shape}")
print(f"Aligned X_test Shape:  {X_test.shape}")
print(f"Aligned X_wls Shape:   {X_wls.shape}")

Aligned X_train Shape: (112, 3)
Aligned X_val Shape:   (24, 3)
Aligned X_test Shape:  (25, 3)
Aligned X_wls Shape:   (40, 3)


In [42]:
# Using TF-IDF baseline vectorizer
## memory: 161 transcripts are just short strings of text. In terms of computer memory, the entire dataset is only a few kilobytes.

# scikit-learn library (which powers TfidfVectorizer) is designed to run exclusively on the CPU

from sklearn.feature_extraction.text import TfidfVectorizer

print("\nExtracting linguistic features...")

# Initialize TF-IDF
# REMOVED stop_words='english' to preserve clinical markers (pronouns, filler words).
# ADDED ngram_range=(1,2) to capture word repetitions and short phrasing markers.
tfidf = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))

# Fit ONLY on training data to prevent data leakage, then transform all datasets
# 1. Process Training Data
num_train = len(X_train['transcript'])
print(f"Processing {num_train} training transcripts...")
X_train_text = tfidf.fit_transform(X_train['transcript']).toarray()
print("  [Done] Training features extracted.")

# 2. Process Validation Data
num_val = len(X_val['transcript'])
print(f"Processing {num_val} validation transcripts...")
X_val_text = tfidf.transform(X_val['transcript']).toarray()
print("  [Done] Validation features extracted.")

# 3. Process Testing Data
num_test = len(X_test['transcript'])
print(f"Processing {num_test} testing transcripts...")
X_test_text = tfidf.transform(X_test['transcript']).toarray()
print("  [Done] Testing features extracted.")

# 4. Process WLS Data
num_wls = len(X_wls['transcript'])
print(f"Processing {num_wls} WLS transcripts...")
X_wls_text = tfidf.transform(X_wls['transcript']).toarray()
print("  [Done] WLS features extracted.")


Extracting linguistic features...
Processing 112 training transcripts...
  [Done] Training features extracted.
Processing 24 validation transcripts...
  [Done] Validation features extracted.
Processing 25 testing transcripts...
  [Done] Testing features extracted.
Processing 40 WLS transcripts...
  [Done] WLS features extracted.


Verify the extracted features:

In [43]:
# 1. Check the dimensions of the new arrays (Rows = Patients, Columns = Features)
print("\n--- Summary ---")
print(f"Success! Extracted {X_train_text.shape[1]} unique word features.")
print(f"Final Training Shape:   {X_train_text.shape} (Patients, Features)")
print(f"Final Validation Shape: {X_val_text.shape} (Patients, Features)")
print(f"Final Testing Shape:    {X_test_text.shape} (Patients, Features)")
print(f"Final WLS Shape:        {X_wls_text.shape} (Patients, Features)")


--- Summary ---
Success! Extracted 1000 unique word features.
Final Training Shape:   (112, 1000) (Patients, Features)
Final Validation Shape: (24, 1000) (Patients, Features)
Final Testing Shape:    (25, 1000) (Patients, Features)
Final WLS Shape:        (40, 1000) (Patients, Features)


In [44]:
# 2. Extract the vocabulary words the vectorizer learned
feature_names = tfidf.get_feature_names_out()
print(f"\nTotal unique words/n-grams kept: {len(feature_names)}")


Total unique words/n-grams kept: 1000


In [45]:
# 3. Print a sample of the vocabulary (e.g., the first 100 words)
print("\nSample of extracted features (alphabetical):")
print(feature_names[:100])


Sample of extracted features (alphabetical):
['about' 'about all' 'about to' 'accident' 'action' 'after' 'ah' 'air'
 'all' 'all over' 'all right' 'all see' 'all the' 'almost' 'already'
 'alright' 'also' 'am' 'an' 'an accident' 'an apron' 'and' 'and another'
 'and blouse' 'and don' 'and drying' 'and girl' 'and he' 'and her'
 'and his' 'and in' 'and it' 'and little' 'and looks' 'and over'
 'and plate' 'and she' 'and that' 'and the' 'and then' 'and there'
 'and they' 'and this' 'and two' 'and uh' 'and um' 'and what' 'another'
 'another one' 'another window' 'any' 'anything' 'anything else' 'apron'
 'are' 'are curtains' 'are getting' 'are the' 'are two' 'arm' 'around'
 'around the' 'as' 'as he' 'as the' 'as though' 'asking' 'asking for' 'at'
 'at it' 'at the' 'attention' 'attention to' 'back' 'back of' 'back the'
 'back to' 'bad' 'be' 'be blowing' 'be in' 'be quiet' 'be the' 'because'
 'because he' 'because she' 'because the' 'before' 'behind' 'being' 'big'
 'blowing' 'boy' 'boy and' 'boy

### Saving the Extracted Linguistic Features in GCS

In [46]:
# Helper for saving and tracking TEXT uploads
uploads_text = []

print("--- Saving and Uploading Linguistic Features (Librosa Aligned) ---")

def save_and_queue_text(name, text_npy, dataset_prefix):
    # Save Text NPY
    text_path = f"local_data/features/librosa/{name}_text.npy"
    np.save(text_path, text_npy)
    uploads_text.append((text_path, f"{dataset_prefix}/features/librosa/{name}_text.npy"))


# Queue ADDReSSo Text
save_and_queue_text("X_train", X_train_text, "ADDReSS2021")
save_and_queue_text("X_val", X_val_text, "ADDReSS2021")
save_and_queue_text("X_test", X_test_text, "ADDReSS2021")

# Queue WLS Text
save_and_queue_text("X_wls", X_wls_text, "WLS")

# Upload all text files
for local_file, gcs_path in uploads_text:
    upload_feature_to_gcs(local_file, gcs_path)

print("\nSuccess! Linguistic features have been uploaded to GCS.")

--- Saving and Uploading Linguistic Features (Librosa Aligned) ---
  Uploaded -> ADDReSS2021/features/librosa/X_train_text.npy
  Uploaded -> ADDReSS2021/features/librosa/X_val_text.npy
  Uploaded -> ADDReSS2021/features/librosa/X_test_text.npy
  Uploaded -> WLS/features/librosa/X_wls_text.npy

Success! Linguistic features have been uploaded to GCS.
